In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input, BatchNormalization, Dropout
from tensorflow.keras.callbacks import EarlyStopping
import numpy as np
import re
import scipy.io

# infile_path = 'xI_actual.csv'
# df = pd.read_csv(infile_path)

# # Extract the 'xI_real' and 'xI_imag' columns
# y = df[['xI_real', 'xI_imag']]

# output_mat_file_path = '/home/luke_mccubbin1/python_scripts/0210Dataset/IQ_xItest.mat'


# # Extract the 'xI_real' and 'xI_imag' columns
# df['xI_complex'] = df['xI_real'] + 1j * df['xI_imag']

# # Convert to NumPy array
# data_to_save = {'xI_actual': df['xI_complex'].values}

# # Save to .mat file
# scipy.io.savemat(output_mat_file_path, data_to_save)

# print(f"Data successfully saved to {output_mat_file_path}")


infile_path = 'Input10dB.csv'
df = pd.read_csv(infile_path)

# Extract the 'xI_real' and 'xI_imag' columns
y = df[['xI_real', 'xI_imag']]

output_mat_file_path = '/home/luke_mccubbin1/python_scripts/0306Dataset/Predist_memory_input10db.mat'


# Extract the 'xI_real' and 'xI_imag' columns
df['xI_complex'] = df['xI_real'] + 1j * df['xI_imag']

# Convert to NumPy array
data_to_save = {'MatlabInput': df['xI_complex'].values}

# Save to .mat file
scipy.io.savemat(output_mat_file_path, data_to_save)

print(f"Data successfully saved to {output_mat_file_path}")


# #Load the training data
outfile_path = 'VloadMemTrain.csv'
df = pd.read_csv(outfile_path)

X = df[['yout_real', 'yout_imag']]


output_mat_file_path = '/home/luke_mccubbin1/python_scripts/0306Dataset/Predist_memory_output10db.mat'


# Extract the 'xI_real' and 'xI_imag' columns
df['yout_complex'] = df['yout_real'] + 1j * df['yout_imag']

# Convert to NumPy array
data_to_save = {'MatlabOuttrain': df['yout_complex'].values}

# Save to .mat file
scipy.io.savemat(output_mat_file_path, data_to_save)

print(f"Data successfully saved to {output_mat_file_path}")


print("Shape of y:", y.shape)
print("Shape of X:", X.shape)

print('NN input files recieved')

Data successfully saved to /home/luke_mccubbin1/python_scripts/0306Dataset/Predist_memory_input10db.mat
Data successfully saved to /home/luke_mccubbin1/python_scripts/0306Dataset/Predist_memory_output10db.mat
Shape of y: (90001, 2)
Shape of X: (90001, 2)
NN input files recieved


In [6]:
#NN model

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)



model = Sequential([
    Input(shape=(2,)),
    Dense(32, activation='relu'),
    Dense(128, activation='relu'),
    Dense(64, activation='relu'),
    Dense(2)
])

# Compile the model
model.compile(optimizer='adam', loss='mean_squared_error', metrics=['accuracy'])

# Train the model
history = model.fit(X_train, y_train, epochs=5, validation_split=0.2, verbose=1)

# Evaluate the model
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=1)


y_pred = model.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)

print(f'Test Loss: {test_loss}')
print(f'Test accuracy: {test_accuracy}')
print(f'Test RMSE: {rmse}')

print("Training complete")


# Function to parse complex numbers, including those with scientific notation from excel csv file
def parse_complex_number(s):
    s = s.replace("i", "j")  
    # Updated regex to handle scientific notation
    match = re.match(r"([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)([-+]\d*\.?\d+(?:[eE][-+]?\d+)?)j", s.replace(" ", ""))
    if match:
        real_part = float(match.group(1))
        imag_part = float(match.group(2))
        return real_part, imag_part
    else:
        raise ValueError(f"Malformed complex number: {s}")

# Load the new CSV file for predictions
# predict_file_path = '/home/luke_mccubbin1/python_scripts/0113Datatest/IQout_data_Pin10dBm_modTest.csv'
# df_predict = pd.read_csv(predict_file_path)

# # Use function to separate the real and imaginary parts of the yout complex numbers
# df_predict[['yout_real', 'yout_imag']] = df_predict['yout'].apply(
#     lambda x: pd.Series(parse_complex_number(x))
# )

# Load the training data
#outfile_path = 'IQout_data_Pin10dBm_mod.csv'
outfile_path = 'Mem22dB_predist.csv'
df = pd.read_csv(outfile_path)

# Prepare input for prediction
X_new = df[['xI_real', 'xI_imag']]

# Predict using the model
y_new_pred = model.predict(X_new)  # Ensure this returns a 2D array

# Create the DataFrame with predictions
df_predict = pd.DataFrame({
    'xI_predict_real': y_new_pred[:, 0],
    'xI_predict_imag': y_new_pred[:, 1]
})

# Create complex numbers
df_predict['xI_predict'] = (df_predict['xI_predict_real'] + 1j * df_predict['xI_predict_imag']).astype(np.complex128)

# Prepare data for MATLAB file generation
data_to_save = {'xI_predict': df_predict['xI_predict'].values}

output_csv_file_path = '/home/luke_mccubbin1/python_scripts/0306Dataset/Mem7dB_predist_predictions.csv'
output_mat_file_path = '/home/luke_mccubbin1/python_scripts/0306Dataset/Mem7dB_predist_predictions.mat'

df_predict[['xI_predict']].to_csv(output_csv_file_path, index=False)
scipy.io.savemat(output_mat_file_path, data_to_save)

print(f'Predictions saved to {output_mat_file_path}')

Epoch 1/5
1800/1800 ━━━━━━━━━━━━━━━━━━━━ 3s 1ms/step - accuracy: 0.9720 - loss: 0.0092 - val_accuracy: 0.9872 - val_loss: 0.0019
Epoch 2/5
1800/1800 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.9849 - loss: 0.0014 - val_accuracy: 0.9818 - val_loss: 0.0013
Epoch 3/5
1800/1800 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.9861 - loss: 0.0013 - val_accuracy: 0.9874 - val_loss: 0.0012
Epoch 4/5
1800/1800 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.9859 - loss: 0.0013 - val_accuracy: 0.9842 - val_loss: 0.0014
Epoch 5/5
1800/1800 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.9856 - loss: 0.0012 - val_accuracy: 0.9867 - val_loss: 0.0012
563/563 ━━━━━━━━━━━━━━━━━━━━ 1s 801us/step - accuracy: 0.9871 - loss: 0.0012
563/563 ━━━━━━━━━━━━━━━━━━━━ 0s 723us/step
Test Loss: 0.0012012944789603353
Test accuracy: 0.9862785339355469
Test RMSE: 0.03465969537011961
Training complete
938/938 ━━━━━━━━━━━━━━━━━━━━ 1s 678us/step
Predictions saved to /home/luke_mccubbin1/python_scripts/0306Dataset/Mem7dB_